In [1]:
# Run this block before getting started.
# It loads the necessary packages, prepares the dataset, and initializes a pre-trained model.

# Packages needed for dataset loading and preprocessing
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Subset, DataLoader

# Packages needed for project tasks (model definition, optimization, distributed training, and evaluation)
import torch
import torch.nn as nn
import torch.optim as optim
from accelerate import Accelerator
from transformers import AutoModelForImageClassification, AutoImageProcessor
import evaluate

# Hiding long output messages
from transformers import logging
logging.set_verbosity_error()

# Setup accelerator
accelerator = Accelerator()

# Prepare CIFAR-10 dataset
# CIFAR-10 is RGB -> 3-channel normalize
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
# Load and reduce CIFAR-10 dataset (only 50 images for faster training)
dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
num_train = 1000
num_val = 200
subset_train = Subset(dataset, range(num_train))
subset_val = Subset(dataset, range(num_train, num_train + num_val))
dataloader = DataLoader(subset_train, batch_size=32, shuffle=True)
val_dataloader = DataLoader(subset_val, batch_size=32)

# Load the pre-trained model and processor
model_name = "microsoft/swin-tiny-patch4-window7-224"
processor = AutoImageProcessor.from_pretrained(model_name)
distributed_model = AutoModelForImageClassification.from_pretrained(
    model_name, num_labels=10, ignore_mismatched_sizes=True
)
initial_state_dict = distributed_model.state_dict().copy()
# del distributed_model # free up memory

/home/athapar/personal/ai-bio-projects/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/athapar/personal/ai-bio-projects/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
Loading weights: 100%|██████████| 233/233 [00:00<00:00, 15678.51it/s]


In [ ]:
criterion = nn.CrossEntropyLoss()
num_epochs = 5
optimizer_summary = {
    "adam": lambda params:  optim.Adam(params,lr=0.001, weight_decay=1e-4),
    "adafactor": lambda params: optim.Adafactor(params,lr=0.001, weight_decay=1e-4),
    "adamw": lambda params: optim.AdamW(params,lr=0.001, weight_decay=1e-4),
    "sgd": lambda params: optim.SGD(params, lr=0.01, momentum=0.0)
}

In [ ]:
import time

training_report = {}
best_optimizer = None # store as {name, loss, accuracy}

print("Training & eval on device:", accelerator.device)
for o_name, opt_fn in optimizer_summary.items():
    start_time = time.time()
     # fresh model each run, same initial weights
    model = AutoModelForImageClassification.from_pretrained(
        model_name, num_labels=10, ignore_mismatched_sizes=True
    )
    model.load_state_dict(initial_state_dict) # start with inital weights for each optimizer
    model, optimizer, data, val_data = accelerator.prepare(model, opt_fn(model.parameters()), dataloader, val_dataloader)

    last_train_loss = 0.0
    # go through epochs
    for epoch in range(1, num_epochs + 1):
        running_train_loss = 0
        #train
        model.train()
        for source, targets in data:
            optimizer.zero_grad()
            outputs = model(pixel_values=source).logits
            loss = criterion(outputs, targets)
            accelerator.backward(loss)
            optimizer.step()
            running_train_loss += loss.item()

        #eval
        metric = evaluate.load("accuracy")
        model.eval()
        for source, targets in val_data:
            with torch.no_grad():
                output = model(pixel_values=source).logits
                preds = output.argmax(dim=1)
                preds, refs = accelerator.gather_for_metrics((preds, targets))
                metric.add_batch(predictions=preds,references=refs)
        accuracy = metric.compute()["accuracy"]

        last_train_loss = running_train_loss / len(data)
        print(f"Epoch {epoch}/{num_epochs} - Optimizer: {o_name} - Loss: {last_train_loss:.4f} - Accuracy: {accuracy:.4f}")

    # after running all epochs
    training_report[o_name] = {
        "training_time": time.time() - start_time,
        "loss": last_train_loss,
        "accuracy": accuracy
    }
    print(
        f"Optimizer: {o_name} - "
        f"Training Time: {training_report[o_name]['training_time']:.2f}s - "
        f"Loss: {last_train_loss:.4f} - Accuracy: {accuracy:.4f}"
    )
    if best_optimizer is None or best_optimizer["accuracy"] < accuracy:
        best_optimizer = {
            "name": o_name,
            "loss": last_train_loss,
            "accuracy": accuracy,
        }
        torch.save(model.state_dict(), "best_model.pth")

print("Best optimizer:", best_optimizer)
print("Training report:", training_report)

Training & eval on device: cuda


Loading weights: 100%|██████████| 233/233 [00:00<00:00, 12288.88it/s]


Epoch 1/5 - Optimizer: adam - Loss: 2.3842 - Accuracy: 0.0900
Epoch 2/5 - Optimizer: adam - Loss: 2.3334 - Accuracy: 0.1050
Epoch 3/5 - Optimizer: adam - Loss: 2.3294 - Accuracy: 0.1050
Epoch 4/5 - Optimizer: adam - Loss: 2.3238 - Accuracy: 0.0900
Epoch 5/5 - Optimizer: adam - Loss: 2.3165 - Accuracy: 0.0750
Optimizer: adam - Training Time: 38.03s - Loss: 2.3165 - Accuracy: 0.0750


Loading weights: 100%|██████████| 233/233 [00:00<00:00, 11132.32it/s]


Epoch 1/5 - Optimizer: adafactor - Loss: 1.1342 - Accuracy: 0.8550
Epoch 2/5 - Optimizer: adafactor - Loss: 0.2384 - Accuracy: 0.8550
Epoch 3/5 - Optimizer: adafactor - Loss: 0.0647 - Accuracy: 0.9400
Epoch 4/5 - Optimizer: adafactor - Loss: 0.0395 - Accuracy: 0.9400
Epoch 5/5 - Optimizer: adafactor - Loss: 0.0126 - Accuracy: 0.9300
Optimizer: adafactor - Training Time: 45.49s - Loss: 0.0126 - Accuracy: 0.9300


Loading weights: 100%|██████████| 233/233 [00:00<00:00, 5216.82it/s]


Epoch 1/5 - Optimizer: adamw - Loss: 2.3795 - Accuracy: 0.0950
Epoch 2/5 - Optimizer: adamw - Loss: 2.3409 - Accuracy: 0.0900
Epoch 3/5 - Optimizer: adamw - Loss: 2.3381 - Accuracy: 0.0600
Epoch 4/5 - Optimizer: adamw - Loss: 2.3249 - Accuracy: 0.1000
Epoch 5/5 - Optimizer: adamw - Loss: 2.3202 - Accuracy: 0.0750
Optimizer: adamw - Training Time: 48.91s - Loss: 2.3202 - Accuracy: 0.0750


Loading weights: 100%|██████████| 233/233 [00:00<00:00, 8644.45it/s]


Epoch 1/5 - Optimizer: sgd - Loss: 1.4356 - Accuracy: 0.8000
Epoch 2/5 - Optimizer: sgd - Loss: 0.4575 - Accuracy: 0.7600
Epoch 3/5 - Optimizer: sgd - Loss: 0.2835 - Accuracy: 0.6900
Epoch 4/5 - Optimizer: sgd - Loss: 0.1252 - Accuracy: 0.9000
Epoch 5/5 - Optimizer: sgd - Loss: 0.0788 - Accuracy: 0.8950
Optimizer: sgd - Training Time: 49.66s - Loss: 0.0788 - Accuracy: 0.8950
Best optimizer: {'name': 'adafactor', 'loss': 0.01262460010946684, 'accuracy': 0.93}
Training report: {'adam': {'training_time': 38.02641844749451, 'loss': 2.3165198961893716, 'accuracy': 0.075}, 'adafactor': {'training_time': 45.48560619354248, 'loss': 0.01262460010946684, 'accuracy': 0.93}, 'adamw': {'training_time': 48.90858316421509, 'loss': 2.32018622519478, 'accuracy': 0.075}, 'sgd': {'training_time': 49.65760946273804, 'loss': 0.0787788739886194, 'accuracy': 0.895}}


In [ ]:
from transformers import TrainingArguments, Trainer
trainer =  Trainer(
    model=model,

)

In [16]:
optimizer = optim.AdamW(params=model.parameters())
optimizer_state = optimizer.state.values()
print(optimizer_state)
def compute_optimizer_size(optimizer_state):
    total_size_megabytes, total_num_elements = 0, 0
    for params in optimizer_state:
        for name, tensor in params.items():
            tensor = torch.tensor(tensor)
            num_elements = tensor.numel()
            element_size = tensor.element_size()
            total_num_elements += num_elements
            total_size_megabytes += num_elements * element_size / (1024 ** 2)
            return total_size_megabytes, total_num_elements
# opt_size_mb, opt_num_elements = compute_optimizer_size(optimizer_state)
# print(f"Optimizer size: {opt_size_mb:.2f} MB - Total elements: {opt_num_elements}")

dict_values([])


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer_configs = {
    "SGD": lambda p: optim.SGD(p, lr=0.01, momentum=0.9, weight_decay=0.0),
    "Adam": lambda p: optim.Adam(p, lr=0.001, weight_decay=1e-4),
    "AdamW": lambda p: optim.AdamW(p, lr=0.001, weight_decay=1e-4),
}
optimizer_summary = {}
training_report = {}
best_optimizer = None
best_accuracy = 0.0
num_epochs = 2

# Evaluate model accuracy using the Hugging Face evaluate library
def evaluate_model(model, dataloader):
    metric = evaluate.load("accuracy")
    model.eval()
    with torch.no_grad():
        for inputs, labels in dataloader:
            outputs = model(pixel_values=inputs).logits
            preds = outputs.argmax(dim=1)
            preds, labels = accelerator.gather_for_metrics((preds, labels))
            metric.add_batch(predictions=preds, references=labels)
    return metric.compute()["accuracy"]

# Train and evaluate the model for each optimizer
for name, opt_fn in optimizer_configs.items():
    model = AutoModelForImageClassification.from_pretrained(
        model_name, num_labels=10, ignore_mismatched_sizes=True
    )
    model.load_state_dict(initial_state_dict)
    optimizer = opt_fn(model.parameters())
    model, optimizer, data = accelerator.prepare(model, optimizer, dataloader)
    model.train()
    for _ in range(num_epochs):
        for inputs, labels in data:
            outputs = model(pixel_values=inputs).logits
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            accelerator.backward(loss)
            optimizer.step()
#   trained_model = model (optional)
    accuracy = evaluate_model(model, data)

#    Save full model (optional)
#    path = f"./trained_model_{name}.pth"
#    torch.save(model, path)

    # Store optimizer configuration and performance results
    if name == "SGD":
        optimizer_summary[name] = {"learning_rate": 0.01, "weight_decay": 0.0}
    elif name == "Adam":
        optimizer_summary[name] = {"learning_rate": 0.001, "weight_decay": 1e-4}
    elif name == "AdamW":
        optimizer_summary[name] = {"learning_rate": 0.001, "weight_decay": 1e-4}

    training_report[name] = {
        "accuracy": accuracy,
        "training_time": 1.0
        #"model_path": path (optional)
    }

    # Select best optimizer based on highest accuracy after 2 epochs
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_optimizer = name

# Add comparison results to the training report
training_report["comparison"] = {
    "best_optimizer": best_optimizer
}

# Print summary of training results
print("\nTraining Report Summary")
for name in optimizer_configs:
    r = training_report[name]
    print(f"\n{name} Optimizer:\n- Accuracy: {r['accuracy']*100:.2f}")
print(f"\nBest Optimizer after {num_epochs} epochs: {training_report['comparison']['best_optimizer']}")

# Note: Due to the very small training subset (only 50 images), the model is unlikely to achieve high accuracy or learn meaningful patterns. This setup is intentionally lightweight for the purposes of this project. You can experiment with a larger dataset and more training epochs after submitting the project to achieve improved and more realistic performance results and compare parameter sizes and training speeds

In [ ]:
import numpy as np

# create 4d array
arr = np.random.rand(2, 3,)




array([[0.38994467, 0.524067  , 0.59122006],
       [0.22456683, 0.32115734, 0.36124767]])